<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 지식 증류 · 모델 경량화 · 01. FitNets

## FitNets: Hints for Thin Deep Nets

- **원 논문:** [FitNets: Hints for Thin Deep Nets](https://arxiv.org/abs/1412.6550)
- **저자 / 발표:** Adriana Romero et al. · ICLR 2015 (2015)
- **난이도 / 예상 시간:** 중급 · 약 75분
- **선수 지식:** PyTorch tensor와 autograd, softmax·cross entropy, 선형대수 기초
- **로컬 학습 데이터:** `data/field_curriculum/compression_bench.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** teacher 중간 표현을 regressor로 맞춘 뒤 출력 distillation을 수행하는 2단계 학습을 구현한다.

**축소하거나 생략한 부분:** 원 논문은 CIFAR/SVHN의 깊고 얇은 CNN을 사용한다. 실습은 32차원 MLP hidden hint로 같은 훈련 순서를 검증한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.1, Eq. (2): hint-based training | Task 1–2 | guided→hint regressor shape/MSE |
| Figure 1: two-stage training | Task 2–3 | hint 선학습 뒤 같은 sequence의 teacher logit으로 분류 |
| §3.2, Eq. (3): knowledge distillation | Task 3 | soft output loss |
| §4: thin/deep student experiments | Task 3 | loss와 accuracy plot |

## 미니 재현
최종 출력만 흉내 내기 어려운 얇은 student에게 teacher의 **중간 표현을 먼저 힌트**로 줍니다.

## 포트폴리오 해설: 정확도와 효율의 계약

$$\mathcal L_{hint}=\frac12\left\|u_h(\mathbf x,W_{Hint})-r(v_g(\mathbf x,W_{Guided}),W_r)
\right\|_2^2$$

- **기호 정의:** u_h는 teacher hint, v_g는 student guided layer, r은 shape regressor입니다.
- **수식의 역할:** 얇은 student가 최종 logit 전에 teacher의 중간 표현을 따라가도록 단계별로 지도합니다.
- **구현 이유:** 정확도만 남기면 경량화의 비용 절감 주장을 검증할 수 없으므로,
  핵심 변환과 품질·비용 측정을 독립 함수와 method로 분리했습니다.
- **Task/코드 대응:** Task P의 논문 전용 class와 `forward`, `compute_loss`,
  `training_step`, `fit_portfolio_model`, `evaluate_portfolio_model`이
  §3.1, Eq. (2): hint-based training의 흐름을 명시합니다.
- **입출력 shape:** token embedding [B, 6, 16] → guided [B, 6, H] → regressed hint [B, 6, 24]
- **평가:** hint MSE, task CE, test accuracy
- **원문 대비 한계:** 원 논문은 CIFAR/SVHN의 깊고 얇은 CNN을 사용한다. 실습은 32차원 MLP hidden hint로 같은 훈련 순서를 검증한다.

경량화 실험에서는 정확도만 보지 않고 parameter·sparsity·bit·latency 중 논문이
주장한 비용을 함께 기록해야 합니다. 아래 코드는 단계별 중간값을 남깁니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** input [B,D] → teacher/student hint [B,Hₜ/Hₛ] → logits [B,K]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from pathlib import Path
import math
import time
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
torch.set_default_device(DEVICE)
print(ACCELERATOR.summary())


def locate(relative):
    for base in (Path.cwd(), *Path.cwd().parents):
        path = base / relative
        if path.exists():
            return path
    raise FileNotFoundError(relative)


dataset_path = locate("data/field_curriculum/compression_bench.npz")
raw = np.load(dataset_path, allow_pickle=False)
features = torch.from_numpy(raw["features"]).float().to(DEVICE)
labels = torch.from_numpy(raw["labels"]).long().to(DEVICE)
teacher_logits = torch.from_numpy(raw["teacher_logits"]).float().to(DEVICE)
teacher_weights = torch.from_numpy(raw["teacher_weights"]).float().to(DEVICE)
calibration = torch.from_numpy(raw["calibration_activations"]).float().to(DEVICE)
dense_weights = torch.from_numpy(raw["dense_weights"]).float().to(DEVICE)
train_idx = torch.from_numpy(raw["train_idx"]).long().to(DEVICE)
test_idx = torch.from_numpy(raw["test_idx"]).long().to(DEVICE)
images = torch.from_numpy(raw["images"]).float().to(DEVICE)
image_labels = torch.from_numpy(raw["image_labels"]).long().to(DEVICE)
image_train_idx = torch.from_numpy(raw["image_train_idx"]).long().to(DEVICE)
image_test_idx = torch.from_numpy(raw["image_test_idx"]).long().to(DEVICE)
token_embeddings = torch.from_numpy(raw["token_embeddings"]).float().to(DEVICE)
teacher_hidden = torch.from_numpy(raw["teacher_hidden"]).float().to(DEVICE)
teacher_attention = torch.from_numpy(raw["teacher_attention"]).float().to(DEVICE)
candidate_configs = torch.from_numpy(raw["candidate_configs"]).float()
assert features.shape == (512, 32) and teacher_logits.shape == (512, 4)
assert images.shape == (320, 1, 16, 16)
assert teacher_hidden.shape == (256, 6, 24)
print(
    dataset_path.name, "features", tuple(features.shape), "images", tuple(images.shape)
)

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §3.1, Eq. (2): hint-based training
- **이 셀의 책임:** Task 1–2
- **Tensor shape 계약:** input [B,D] → teacher/student hint [B,Hₜ/Hₛ] → logits [B,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** guided→hint regressor shape/MSE. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
class FitStudent(nn.Module):
    def __init__(self):
        super().__init__()
        self.guided = nn.Linear(32, 12)
        self.regressor = nn.Linear(12, 24)
        self.head = nn.Linear(12, 4)

    def forward(self, x):
        h = F.relu(self.guided(x))
        return self.head(h), self.regressor(h)


student = FitStudent()
teacher_hint = torch.cat([features[:, :16], features[:, 16:24].square()], dim=1)
logits, hint = student(features[:5])
assert logits.shape == (5, 4) and hint.shape == teacher_hint[:5].shape

### 구현 단계 3 · 목적함수·학습 update

- **원문 위치:** §3.1, Eq. (2): hint-based training / Figure 1: two-stage training
- **이 셀의 책임:** Task 1–2 / Task 2–3
- **Tensor shape 계약:** input [B,D] → teacher/student hint [B,Hₜ/Hₛ] → logits [B,K]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** guided→hint regressor shape/MSE / hint 선학습 뒤 같은 sequence의 teacher logit으로 분류. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
hint_optimizer = torch.optim.Adam(
    list(student.guided.parameters()) + list(student.regressor.parameters()), lr=0.025
)
hint_losses = []
for _ in range(50):
    hint_optimizer.zero_grad()
    _, predicted_hint = student(features[train_idx])
    loss = F.mse_loss(predicted_hint, teacher_hint[train_idx])
    loss.backward()
    hint_optimizer.step()
    hint_losses.append(float(loss.detach()))
assert hint_losses[-1] < hint_losses[0] * 0.55
guided_before = student.guided.weight.detach().clone()

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** Figure 1: two-stage training / §3.2, Eq. (3): knowledge distillation / §4: thin/deep student experiments
- **이 셀의 책임:** Task 2–3 / Task 3
- **Tensor shape 계약:** input [B,D] → teacher/student hint [B,Hₜ/Hₛ] → logits [B,K]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** hint 선학습 뒤 같은 sequence의 teacher logit으로 분류 / soft output loss / loss와 accuracy plot. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
optimizer = torch.optim.Adam(student.parameters(), lr=0.02)
task_losses = []
for _ in range(50):
    optimizer.zero_grad()
    logits, _ = student(features[train_idx])
    loss = 0.65 * F.kl_div(
        F.log_softmax(logits / 3, -1),
        F.softmax(teacher_logits[train_idx] / 3, -1),
        reduction="batchmean",
    ) * 9 + 0.35 * F.cross_entropy(logits, labels[train_idx])
    loss.backward()
    optimizer.step()
    task_losses.append(float(loss.detach()))
with torch.no_grad():
    accuracy = (
        (student(features[test_idx])[0].argmax(-1) == labels[test_idx])
        .float()
        .mean()
        .item()
    )
assert (
    task_losses[-1] < task_losses[0]
    and accuracy > 0.8
    and not torch.equal(student.guided.weight, guided_before)
)
plt.plot(hint_losses, label="hint")
plt.plot(task_losses, label="task KD")
plt.legend()
plt.show()
print(f"student accuracy={accuracy:.1%}")

### Task P1 · 압축 대상과 핵심 모델

tensor shape와 비용 단위를 먼저 확인한 뒤 논문 전용 class의 `forward`와
`compute_loss`에 정확도·표현 보존 계약을 명시합니다.

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** §3.1, Eq. (2): hint-based training / Figure 1: two-stage training / §3.2, Eq. (3): knowledge distillation / §4: thin/deep student experiments
- **이 셀의 책임:** Task 1–2 / Task 2–3 / Task 3
- **Tensor shape 계약:** input [B,D] → teacher/student hint [B,Hₜ/Hₛ] → logits [B,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** guided→hint regressor shape/MSE / hint 선학습 뒤 같은 sequence의 teacher logit으로 분류 / soft output loss / loss와 accuracy plot. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    hidden_dim: int = 12
    hint_weight: float = 1.0
    temperature: float = 2.0
    learning_rate: float = 0.025
    hint_steps: int = 8
    distillation_steps: int = 8


def prepare_batch() -> tuple[Tensor, Tensor, Tensor, Tensor]:
    inputs = token_embeddings[:192]
    targets, predictions = sequence_teacher_targets(inputs)
    hints = teacher_hidden[:192]
    return inputs, targets, hints, predictions


def sequence_teacher_targets(inputs: Tensor) -> tuple[Tensor, Tensor]:
    """Create labels and soft logits from the same sequence seen by the student."""
    class_count = teacher_logits.shape[-1]
    pooled = inputs.mean(dim=1)
    predictions = 4.0 * pooled[:, :class_count]
    return predictions.argmax(dim=1), predictions


class FitnetsPortfolioModel(nn.Module):
    def __init__(self, config: PortfolioConfig):
        super().__init__()
        self.config = config
        self.guided = nn.Linear(token_embeddings.shape[-1], config.hidden_dim)
        self.regressor = nn.Linear(config.hidden_dim, teacher_hidden.shape[-1])
        self.classifier = nn.Linear(config.hidden_dim, teacher_logits.shape[-1])

    def forward(self, inputs: Tensor) -> tuple[Tensor, Tensor]:
        guided = F.relu(self.guided(inputs))
        regressed_hint = self.regressor(guided)
        logits = self.classifier(guided.mean(dim=1))
        return logits, regressed_hint

    def compute_hint_loss(
        self,
        outputs: tuple[Tensor, Tensor],
        teacher_hint: Tensor,
    ) -> Tensor:
        _, regressed_hint = outputs
        return self.config.hint_weight * F.mse_loss(regressed_hint, teacher_hint)

    def compute_distillation_loss(
        self,
        outputs: tuple[Tensor, Tensor],
        targets: Tensor,
        teacher_prediction: Tensor,
    ) -> Tensor:
        logits, _ = outputs
        temperature = self.config.temperature
        soft_targets = torch.softmax(teacher_prediction / temperature, dim=1)
        soft_loss = F.kl_div(
            F.log_softmax(logits / temperature, dim=1),
            soft_targets,
            reduction="batchmean",
        )
        return F.cross_entropy(logits, targets) + temperature**2 * soft_loss

    def training_step(
        self,
        inputs: Tensor,
        target: Tensor,
        teacher_target: Tensor,
        phase: str,
    ) -> Tensor:
        outputs = self(inputs)
        if phase == "hint":
            return self.compute_hint_loss(outputs, teacher_target)
        if phase == "distillation":
            return self.compute_distillation_loss(outputs, target, teacher_target)
        raise ValueError(f"unknown FitNets phase: {phase}")

### Task P2 · 학습·pruning·distillation update

`fit_portfolio_model`에서 optimizer, mask/EMA/quantization 경계와 update 순서를
드러냅니다. 비용 절감이 학습 경로에 미치는 영향을 함께 읽으세요.

### 구현 단계 6 · 목적함수·학습 update

- **원문 위치:** §3.1, Eq. (2): hint-based training / Figure 1: two-stage training / §3.2, Eq. (3): knowledge distillation
- **이 셀의 책임:** Task 1–2 / Task 2–3 / Task 3
- **Tensor shape 계약:** input [B,D] → teacher/student hint [B,Hₜ/Hₛ] → logits [B,K]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** guided→hint regressor shape/MSE / hint 선학습 뒤 같은 sequence의 teacher logit으로 분류 / soft output loss. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(model: FitnetsPortfolioModel) -> list[float]:
    inputs, targets, hints, teacher_predictions = prepare_batch()
    hint_parameters = list(model.guided.parameters()) + list(
        model.regressor.parameters()
    )
    hint_optimizer = torch.optim.Adam(
        hint_parameters,
        lr=model.config.learning_rate,
    )
    history = []
    for _ in range(model.config.hint_steps):
        loss = model.training_step(inputs, targets, hints, "hint")
        hint_optimizer.zero_grad()
        loss.backward()
        hint_optimizer.step()
        history.append(float(loss.detach().cpu()))
    model.regressor.requires_grad_(False)
    distillation_optimizer = torch.optim.Adam(
        list(model.guided.parameters()) + list(model.classifier.parameters()),
        lr=model.config.learning_rate,
    )
    for _ in range(model.config.distillation_steps):
        loss = model.training_step(
            inputs,
            targets,
            teacher_predictions,
            "distillation",
        )
        distillation_optimizer.zero_grad()
        loss.backward()
        distillation_optimizer.step()
        history.append(float(loss.detach().cpu()))
    return history

### Task P3 · 품질과 비용을 함께 평가

accuracy 또는 reconstruction 품질과 parameter·sparsity·bit·latency를 별도로
기록합니다. 로컬 지표는 원문 하드웨어 benchmark의 대체값이 아닙니다.

### 구현 단계 7 · 평가·완료 증거

- **원문 위치:** §4: thin/deep student experiments
- **이 셀의 책임:** Task 3
- **Tensor shape 계약:** input [B,D] → teacher/student hint [B,Hₜ/Hₛ] → logits [B,K]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** loss와 accuracy plot. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(model: FitnetsPortfolioModel) -> dict[str, float]:
    with torch.no_grad():
        test_inputs = token_embeddings[192:256]
        test_targets, teacher_predictions = sequence_teacher_targets(test_inputs)
        logits, hint = model(test_inputs)
        accuracy = (logits.argmax(1) == test_targets).float().mean()
        hint_mse = F.mse_loss(hint, teacher_hidden[192:256])
        temperature = model.config.temperature
        teacher_probability = torch.softmax(
            teacher_predictions / temperature,
            dim=1,
        )
        prediction_kl = F.kl_div(
            F.log_softmax(logits / temperature, dim=1),
            teacher_probability,
            reduction="batchmean",
        )
    return {
        "test_accuracy": float(accuracy.cpu()),
        "hint_mse": float(hint_mse.cpu()),
        "teacher_student_kl": float(prediction_kl.cpu()),
        "two_stage_updates": float(
            model.config.hint_steps + model.config.distillation_steps
        ),
    }


portfolio_model = FitnetsPortfolioModel(PortfolioConfig())
portfolio_history = fit_portfolio_model(portfolio_model)
portfolio_metrics = evaluate_portfolio_model(portfolio_model)
assert np.isfinite(portfolio_history).all()
assert portfolio_metrics["test_accuracy"] > 0.35
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 원 논문은 CIFAR/SVHN의 깊고 얇은 CNN을 사용한다. 실습은 32차원 MLP hidden hint로 같은 훈련 순서를 검증한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.